In [51]:
!pip install transformers pandas scikit-learn torch
!pip install sentencepiece


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [52]:
import json
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset

In [53]:
DATA_PATH = r"../datasets/food_dataset_extended.json"  # ensure file path is correct

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)


df = pd.DataFrame(data)


# Ensure key columns exist
required_cols = ["item_name", "cuisine", "occasion"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")


# Handle occasions that are lists
df["occasion_text"] = df["occasion"].apply(lambda x: ", ".join(x) if isinstance(x, list) else str(x))


# Create target text: combine dishes for each occasion
df["target_menu"] = df.groupby("occasion_text")["item_name"].transform(lambda x: ", ".join(set(x)))


# Drop duplicates
df = df[["occasion_text", "cuisine", "target_menu"]].drop_duplicates()


# Prepare input-output pairs
df["input_text"] = "Occasion: " + df["occasion_text"] + " | Cuisine: " + df["cuisine"]
df["target_text"] = df["target_menu"]


# Train-test split
train_df, eval_df = train_test_split(df, test_size=0.1, random_state=42)

In [54]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
from torch.utils.data import Dataset
import torch

# Load tokenizer & model
tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small")

# Tokenize inputs and targets
train_encodings = tokenizer(
    list(train_df["input_text"]),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

label_encodings = tokenizer(
    list(train_df["target_text"]),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# ✅ Fixed Dataset class
class MenuDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels["input_ids"]

    def __len__(self):
        return self.labels.size(0)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

# Create dataset
train_dataset = MenuDataset(train_encodings, label_encodings)


In [55]:
train_dataset = MenuDataset(train_encodings, label_encodings)

print(len(train_dataset))
print(train_dataset[0].keys())  # should show: dict_keys(['input_ids', 'attention_mask', 'labels'])

201
dict_keys(['input_ids', 'attention_mask', 'labels'])


In [56]:
training_args = TrainingArguments(
    output_dir="./menu_generator_model",
    per_device_train_batch_size=4,
    num_train_epochs=10,        # 🔼 increase to 10–15
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=50,
    learning_rate=5e-4          # slightly higher LR helps small data
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

In [57]:
trainer.train()

d:\Projects\SPC\spc\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,1.891300
100,0.870300
150,0.586000
200,0.421200
250,0.358600
300,0.309100
350,0.264100
400,0.261000
450,0.229000
500,0.221500


TrainOutput(global_step=510, training_loss=0.5344332299980463, metrics={'train_runtime': 233.6919, 'train_samples_per_second': 8.601, 'train_steps_per_second': 2.182, 'total_flos': 15408346890240.0, 'train_loss': 0.5344332299980463, 'epoch': 10.0})

In [65]:
model.save_pretrained("./trained_menu_model")
tokenizer.save_pretrained("./trained_menu_model")
print("✅ Model training complete and saved!")

✅ Model training complete and saved!


In [62]:
def generate_menu(occasion, cuisine, model, tokenizer):
    prompt = f"Occasion: {occasion} | Cuisine: {cuisine}"
    inputs = tokenizer(prompt, return_tensors="pt")
    output = model.generate(**inputs, max_length=1000, num_beams=5, temperature=0.8)
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [64]:
test_occasions = [
    ("Corporate Lunch", "Indian (North)"),
    ("Wedding", "Indian (South)"),
    ("Festival", "Indian (North)"),
    ("Birthday Party", "Continental"),
]


for occasion, cuisine in test_occasions:
    menu = generate_menu(occasion, cuisine, model, tokenizer)
    print(f"\n=== {occasion.upper()} ({cuisine}) ===")
    print(menu)


print("\n🎉 Generative menu model ready!")


=== CORPORATE LUNCH (Indian (North)) ===
Chole Bhature

=== WEDDING (Indian (South)) ===
Veg Pulao, Pesarattu

=== FESTIVAL (Indian (North)) ===
Aloo Gobi, Masala Dosa, Chole Bhature

=== BIRTHDAY PARTY (Continental) ===
Paneer Butter Masala

🎉 Generative menu model ready!
